# 04D — Phase 0 Diagnostics: Root-Causing the VQC Accuracy Bug

**Context.** Notebooks 04A (raw features) and 04B (scaled features) trained
the 4-qubit VQC to only ~58% test accuracy on binary MNIST (0 vs 1) — barely
above the 53% majority-class baseline — with training loss plateauing near
ln(2) ≈ 0.693 (random-guessing loss) from epoch 2 of 30 onward. A plain
`sklearn.LogisticRegression` on the *same* 4 PCA features reaches **99.66%**
accuracy (`SVC(rbf)`: 99.9%), so this is not a case of quantum models being
inherently weaker on a hard task — something in the VQC training pipeline
was not working, and every downstream research question (RQ1–RQ5) depends
on this being fixed and understood before proceeding.

This notebook runs the ordered diagnostic protocol from the thesis
architecture plan:

1. **Gradient flow audit** — do all 14 trainable parameters receive
   nonzero gradients?
2. **Optimizer registration check** — is the optimizer tracking every
   parameter `model.parameters()` reports?
3. **Parameter movement tracking** — does the *quantum* branch actually
   move during training, or only the classical head?
4. **Tiny-subset overfit test** — the most diagnostic step: can the
   current architecture overfit 40 well-separated points? This is the
   fork in the road between "optimization-scale problem" (more
   epochs/LR would fix it) and "expressivity problem" (the observable
   itself is a bottleneck).
5. **Observable ablation** (only if step 4 fails) — replace the legacy
   single-qubit `SparsePauliOp("ZIII")` readout with one Z observable
   per qubit (`observable_mode="multi_z"`).

Along the way, a second, independent bug was found and fixed directly in
`src/models/quantum_model.py` (see `tests/test_reproducibility.py`):
`TorchConnector`'s default initial ansatz weights are drawn from
PyTorch's *global* RNG, and `create_model(seed=...)` was only ever
passing `seed` to `StatevectorEstimator` — so two "same seed" model
instantiations actually started from genuinely different random weights.
This is fixed by drawing `initial_weights` from a local
`np.random.Generator(seed)`, confirmed bit-identical output across
instantiations.

In [1]:
from pathlib import Path
import sys

import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.models.quantum_model import create_model, get_output_dim
from src.models.hybrid_classifier import HybridClassifier
from src.diagnostics.gradient_audit import audit_gradients, check_optimizer_registration
from src.diagnostics.param_tracking import ParameterMovementTracker
from src.diagnostics.overfit_test import select_tiny_subset, run_overfit_test

set_seed(42)

print("Project root:", PROJECT_ROOT)

Project root: C:\Work\Quantum-Adversarial-Robustness


## Data

Same binary (0 vs 1) / 4-PCA-feature dataset used by 04A/04B, standardized
with `StandardScaler` (fit on train only).

In [2]:
X_train = np.load(PROJECT_ROOT / "data" / "binary" / "X_train.npy")
y_train = np.load(PROJECT_ROOT / "data" / "binary" / "y_train.npy")
X_test = np.load(PROJECT_ROOT / "data" / "binary" / "X_test.npy")
y_test = np.load(PROJECT_ROOT / "data" / "binary" / "y_test.npy")

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("class balance (train):", y_train.mean(), " (test):", y_test.mean())

X_train: (11824, 4)  X_test: (2956, 4)
class balance (train): 0.5329837618403248  (test): 0.5328146143437077


## Ground truth: is this task actually easy?

A quick classical sanity check, run once at the start of this
investigation, establishes the target the VQC should be able to
approach.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

clf = LogisticRegression(max_iter=1000).fit(X_train_s, y_train.ravel())
print("LogisticRegression accuracy:", clf.score(X_test_s, y_test.ravel()))

clf2 = SVC(kernel="rbf").fit(X_train_s, y_train.ravel())
print("SVC(rbf) accuracy:", clf2.score(X_test_s, y_test.ravel()))

LogisticRegression accuracy: 0.996617050067659
SVC(rbf) accuracy: 0.9989851150202977


## Step 1 — Gradient flow audit

One real forward/backward pass on the legacy `observable_mode="single_z"`
architecture (matching 04A/04B). Expect three parameter groups:
`quantum.weight` (12,), `classifier.weight` (1,1), `classifier.bias` (1,),
all with nonzero gradient norm.

In [4]:
quantum_model = create_model(observable_mode="single_z", seed=42)
model = HybridClassifier(quantum_model, quantum_output_dim=1)

x_batch = torch.tensor(X_train_s[:32], dtype=torch.float32)
y_batch = torch.tensor(y_train[:32], dtype=torch.float32).reshape(-1, 1)
criterion = nn.BCEWithLogitsLoss()

report = audit_gradients(model, criterion, x_batch, y_batch)
for r in report:
    print(r)

{'name': 'quantum.weight', 'shape': (12,), 'grad_is_none': False, 'grad_norm': 0.022005708888173103}
{'name': 'classifier.weight', 'shape': (1, 1), 'grad_is_none': False, 'grad_norm': 0.0071580978110432625}
{'name': 'classifier.bias', 'shape': (1,), 'grad_is_none': False, 'grad_norm': 0.19437889754772186}


**Result:** all three parameter groups receive nonzero gradients.
`classifier.bias` gets a substantially larger gradient norm than
`quantum.weight` (0.19 vs 0.02 in the run below) — an early hint that the
optimizer's easiest path to reduce loss is shifting the output bias
towards the majority class rather than using the quantum signal, which
would produce exactly the "near-majority-class accuracy, loss plateau
near ln(2)" pattern seen in 04A/04B. Not yet conclusive on its own — the
gradients are at least present, so this rules out a fully broken gradient
path, not a training-dynamics problem.

## Step 2 — Optimizer registration check

Confirms the optimizer is tracking every parameter `model.parameters()`
reports (14 total: 12 ansatz weights + classifier weight + bias).

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(check_optimizer_registration(model, optimizer))

{'model_param_count': 14, 'optimizer_param_count': 14, 'match': True}


**Result:** 14/14 parameters registered — rules out a silently frozen parameter.

## Step 4 — Tiny-subset overfit test (single_z, legacy architecture)

The most diagnostic step. 40 samples (20 per class), full-batch gradient
descent, 200 epochs, LR 0.05. If this architecture *can* memorize 40
well-separated points, the 58% full-dataset result is an
optimization-scale problem (LR/epochs), not an expressivity ceiling.

This cell takes several minutes (parameter-shift gradients over
`StatevectorEstimator`, ~1-2s per full-batch step).

In [6]:
X_small, y_small = select_tiny_subset(X_train_s, y_train, n_per_class=20, seed=42)
print("tiny subset:", X_small.shape, "class balance:", y_small.mean())

quantum_model_overfit = create_model(observable_mode="single_z", seed=42)
overfit_model = HybridClassifier(quantum_model_overfit, quantum_output_dim=1)
tracker = ParameterMovementTracker(overfit_model)

history_single_z = run_overfit_test(overfit_model, X_small, y_small, epochs=200, lr=0.05)

for h in history_single_z[::20]:
    print(h)
print("final:", history_single_z[-1])
print()
print("parameter movement after 200 epochs:", tracker.movement(overfit_model))

tiny subset: (40, 4) class balance: 0.5
{'epoch': 0, 'loss': 0.7533987760543823, 'accuracy': 0.5}
{'epoch': 20, 'loss': 0.5489503145217896, 'accuracy': 0.800000011920929}
{'epoch': 40, 'loss': 0.4580755829811096, 'accuracy': 0.9750000238418579}
{'epoch': 60, 'loss': 0.398029625415802, 'accuracy': 0.925000011920929}
{'epoch': 80, 'loss': 0.3552958369255066, 'accuracy': 0.949999988079071}
{'epoch': 100, 'loss': 0.3232780992984772, 'accuracy': 0.9750000238418579}
{'epoch': 120, 'loss': 0.29830777645111084, 'accuracy': 0.9750000238418579}
{'epoch': 140, 'loss': 0.2782227396965027, 'accuracy': 0.9750000238418579}
{'epoch': 160, 'loss': 0.26168254017829895, 'accuracy': 0.9750000238418579}
{'epoch': 180, 'loss': 0.24779972434043884, 'accuracy': 0.9750000238418579}
final: {'epoch': 199, 'loss': 0.23651254177093506, 'accuracy': 0.9750000238418579}

parameter movement after 200 epochs: {'quantum.weight': 2.8020522594451904, 'classifier.weight': 8.035411834716797, 'classifier.bias': 1.10696673393

**Result: overfits cleanly** — 97.5% accuracy on the tiny subset by
epoch ~40, loss dropping smoothly and monotonically the whole way (final
loss 0.237, final accuracy 97.5%). Both `quantum.weight` and
`classifier.weight`/`bias` move substantially from their initial values
(L2 distances of 2.80 / 8.04 / 1.11 respectively) — the quantum branch is
not stuck.

**Conclusion of the fork:** the legacy `single_z` architecture *can*
represent a near-perfect decision boundary for this task. The 58%
full-dataset result is **not** a fundamental expressivity/observable
capacity problem — it is an optimization-scale problem (something about
how training proceeds on the full 11,824-sample dataset with the original
hyperparameters: LR 0.01, 30 epochs, batch size 32). This directly
contradicts the initial top hypothesis (that the single-qubit `"ZIII"`
observable was too weak a readout) — a useful reminder that the tiny-subset
overfit test, not architectural speculation, is what actually settles this
question.

Per the diagnostic protocol, this means the *observable ablation (multi_z)
is not required as the fix* — we proceed to Step 6 (LR/epoch investigation
on the full dataset) rather than Step 5. The multi_z ablation is still run
below for completeness / as a documented ablation, since it was already
in flight, but it is not expected to be necessary.

## Step 4 (ablation, run for completeness) — `multi_z` observable

Same tiny-subset test with `observable_mode="multi_z"` (one Z observable
per qubit, 4-dimensional QNN output feeding a wider classical head).
Included as a documented ablation, not because Step 4 above indicated it
was needed. Run for 60 epochs rather than 200 (single_z already answered
the main question; 60 is enough to see the clear non-plateauing trend
below, and each epoch costs ~35-40s of parameter-shift-gradient
computation).

In [7]:
out_dim = get_output_dim(4, "multi_z")
quantum_model_multi = create_model(observable_mode="multi_z", seed=42)
overfit_model_multi = HybridClassifier(quantum_model_multi, quantum_output_dim=out_dim)
tracker_multi = ParameterMovementTracker(overfit_model_multi)

history_multi_z = run_overfit_test(overfit_model_multi, X_small, y_small, epochs=60, lr=0.05)

for h in history_multi_z[::20]:
    print(h)
print("final:", history_multi_z[-1])
print()
print("parameter movement after 60 epochs:", tracker_multi.movement(overfit_model_multi))

{'epoch': 0, 'loss': 0.7044338583946228, 'accuracy': 0.550000011920929}
{'epoch': 20, 'loss': 0.5305168628692627, 'accuracy': 0.8500000238418579}
{'epoch': 40, 'loss': 0.4213651716709137, 'accuracy': 0.875}
final: {'epoch': 59, 'loss': 0.3591309189796448, 'accuracy': 0.8999999761581421}

parameter movement after 60 epochs: {'quantum.weight': 2.8377671241760254, 'classifier.weight': 4.085965156555176, 'classifier.bias': 0.0054458752274513245}


**Result: `multi_z` also overfits cleanly** — 90.0% accuracy on the
tiny subset by epoch 59, loss dropping smoothly and monotonically
throughout (final loss 0.359). Convergence is a little slower than
`single_z` per epoch (which reached 92.5-97.5% in the same/fewer epochs),
but shows no sign of a plateau — it would very likely continue closing
the gap with more epochs. Parameter movement: `quantum.weight` moves
about as much as in the `single_z` case (2.84 vs 2.80), while
`classifier.bias` moves far less (0.0054 vs 1.11) — with 4 classifier
weights instead of 1, the classical head has more distributed capacity
and relies far less on shifting a single bias term. This confirms the
Step 4 conclusion from a second angle: observable capacity was never the
bottleneck, for either the single-qubit or multi-qubit readout.

## Summary and next step

Both the legacy `single_z` observable and the `multi_z` alternative can
represent a near-perfect decision boundary for this task on a tiny
subset. The 58% full-dataset result is an **optimization-scale problem**
(something about how training proceeds over the full 11,824-sample
dataset with the original hyperparameters — LR 0.01, 30 epochs, batch
size 32), not an architectural/expressivity ceiling. `observable_mode`
stays configurable in `src/models/quantum_model.py` (default remains
`single_z` for backward compatibility with existing checkpoints/04C),
but the Phase 0 fix does not require switching away from it.

Combined with the independently-fixed weight-initialization seeding bug
(see `tests/test_reproducibility.py`) — which by itself may have been a
meaningful contributor to unstable/unlucky training outcomes across the
project's notebooks — the next step (Step 6) is to retrain on the full
dataset with a properly seeded model and revisit LR/epoch count. This is
carried out in `04E_train_all_seeds.ipynb`.